# `bw_timex` — Temporal distributions on **background** exchanges (real premise data)

This notebook is a sibling of [`teaching_example_ev_premise.ipynb`](teaching/teaching_example_ev_premise.ipynb).
It uses the **same project and prospective premise databases**, but instead of putting temporal
distributions (TDs) on the *foreground* edges, we put them on edges **inside the background system**
and let the traversal descend into them with the `traverse_background=True` option.

**The experiment**

1. Take a background process `P1` and add a TD to one of its technosphere exchanges.
2. Go one step further down the supply chain: the *input* of that exchange is `P2`. Add a TD to one
   of `P2`'s exchanges too.
3. Build a tiny foreground process `A` that consumes `P1`.
4. Run a `TimexLCA` with `traverse_background=True` and confirm from the timeline that the descent
   actually reaches `P2` (and `P3`) at the time-shifted, variant-resolved dates.

```mermaid
flowchart LR
subgraph fg[<i>foreground</i>]
  A("Process A"):::fg
end
subgraph bg[<i>background (premise variants)</i>]
  P1("P1: passenger car production,<br/>electric, w/o battery"):::bg
  P2("P2: market for glider,<br/>passenger car"):::bg
  P3("P3: glider production,<br/>passenger car"):::bg
end
A -->|"1 unit"| P1
P1 -->|"0.91 kg <b>+ TD₁</b>"| P2
P2 -->|"1 kg <b>+ TD₂</b>"| P3
classDef fg color:#222832, fill:#9c5ffd, stroke:none;
classDef bg color:#222832, fill:#3fb1c5, stroke:none;
```

## Step 0 — Setup (same project as the EV teaching example)

In [1]:
import bw2data as bd

bd.projects.set_current("ei312_REMIND_EU")

BG_DATABASE = "ei312_REMIND-EU_SSP2_NDC_2020"  # the variant the foreground references

if "foreground" in bd.databases:
    del bd.databases["foreground"]  # rebuild the foreground from scratch
foreground = bd.Database("foreground")
foreground.register()

## Step 1 — Pick a background process `P1` and inspect its exchanges

We grab a *random-ish* but meaningful background process: the production of an electric passenger car
glider-less body. We list its technosphere exchanges and pick one to temporalize.

In [2]:
P1 = bd.get_node(
    database=BG_DATABASE,
    name="passenger car production, electric, without battery",
)
print("P1:", P1["name"], "|", P1.get("location"), "|", P1["code"])
print("\nTechnosphere exchanges of P1:")
for i, exc in enumerate(P1.technosphere()):
    inp = exc.input
    print(f"  [{i}] {exc['amount']:+.4g} {inp['unit']:>10} | {inp['name']} ({inp.get('location')})")

P1: passenger car production, electric, without battery | GLO | 24511137e9da4c9ebb5f39d4caea0705

Technosphere exchanges of P1:
  [0] +0.9126   kilogram | market for glider, passenger car (GLO)
  [1] +1       unit | market for manual dismantling of used electric passenger car (GLO)
  [2] +0.08736   kilogram | market for powertrain, for electric passenger car (GLO)
  [3] -4.626e-07   kilogram | market for waste glass (ZA)
  [4] -1.57e-07   kilogram | market for waste glass (PE)
  [5] -9.193e-05   kilogram | market group for waste glass (RER)
  [6] -1.108e-06   kilogram | market for waste glass (CO)
  [7] -1.034e-07   kilogram | market for waste glass (IN)
  [8] -0.03639   kilogram | market for waste glass (RoW)
  [9] -1.202e-05   kilogram | market for waste glass (BR)
  [10] -2.514e-08   kilogram | market for waste glass (CY)
  [11] -0.002316   kilogram | market for waste mineral oil (RoW)
  [12] -0.001114   kilogram | market for waste mineral oil (Europe without Switzerland)
  [13] -3.

We select exchange `[0]` — the **glider** input (`market for glider, passenger car`, ~0.91 kg).
This is `P1 → P2`. We attach a temporal distribution `TD₁` saying the glider is sourced **1–2 years
before** the car is produced.

In [3]:
from bw_temporalis import TemporalDistribution
import numpy as np

# the exchange P1 -> P2 (glider)
e1 = next(
    exc for exc in P1.technosphere()
    if exc.input["name"] == "market for glider, passenger car"
)
P2 = e1.input
print("e1:", round(e1["amount"], 4), e1.input["name"], "->", e1.output["name"])

td1 = TemporalDistribution(
    date=np.array([-2, -1], dtype="timedelta64[Y]"),
    amount=np.array([0.4, 0.6]),
)
e1["temporal_distribution"] = td1
e1.save()
print("Added TD1 to e1.")

e1: 0.9126 market for glider, passenger car -> passenger car production, electric, without battery
Added TD1 to e1.


## Step 2 — Go one step deeper: add a TD to an exchange of `P2`

`P2` is `market for glider, passenger car`. We inspect *its* exchanges and temporalize the link to
`P3` (`glider production, passenger car`) with `TD₂`: production happens **6–12 months before** the
market delivers.

In [4]:
print("P2:", P2["name"], "|", P2.get("location"), "|", P2["code"])
print("\nTechnosphere exchanges of P2:")
for i, exc in enumerate(P2.technosphere()):
    inp = exc.input
    print(f"  [{i}] {exc['amount']:+.4g} {inp['unit']:>10} | {inp['name']} ({inp.get('location')})")

P2: market for glider, passenger car | GLO | 487a4e1fe9564da4b86aec0b20def778

Technosphere exchanges of P2:
  [0] +1   kilogram | glider production, passenger car (GLO)


In [5]:
# the exchange P2 -> P3 (glider production)
e2 = next(
    exc for exc in P2.technosphere()
    if exc.input["name"] == "glider production, passenger car"
)
P3 = e2.input
print("e2:", round(e2["amount"], 4), e2.input["name"], "->", e2.output["name"])

td2 = TemporalDistribution(
    date=np.array([-12, -9, -6], dtype="timedelta64[M]"),
    amount=np.array([0.3, 0.4, 0.3]),
)
e2["temporal_distribution"] = td2
e2.save()
print("Added TD2 to e2.")

e2: 1.0 glider production, passenger car -> market for glider, passenger car
Added TD2 to e2.


> **Note:** these `temporal_distribution` keys are now persisted on exchanges *inside* the
> premise background database. They are ignored by any normal run (and by `traverse_background=False`),
> so they don't affect the other notebooks. The last cell shows how to remove them again.

## Step 3 — Build a tiny foreground process `A` that consumes `P1`

In [6]:
A = foreground.new_node("A", name="process A", unit="unit")
A["reference product"] = "A"
A.save()

A.new_edge(input=A, amount=1, type="production").save()
A.new_edge(input=P1, amount=1, type="technosphere").save()
print("Foreground A -> P1 created.")

Foreground A -> P1 created.


## Step 4 — `TimexLCA` with `traverse_background=True`

In [7]:
from datetime import datetime

method = (
    "ecoinvent-3.12",
    "EF v3.1",
    "climate change",
    "global warming potential (GWP100)",
)

database_dates = {
    "ei312_REMIND-EU_SSP2_NDC_2020": datetime.strptime("2020", "%Y"),
    "ei312_REMIND-EU_SSP2_NDC_2030": datetime.strptime("2030", "%Y"),
    "ei312_REMIND-EU_SSP2_NDC_2040": datetime.strptime("2040", "%Y"),
    "foreground": "dynamic",
}

In [8]:
from bw_timex import TimexLCA

# This single line runs a full static ecoinvent LCA of the demand under the hood
# (the "base LCA"), which is the slowest step of the whole notebook (~30 s on a
# cold cache). We build exactly ONE TimexLCA and reuse it below.
tlca = TimexLCA({A: 1}, method, database_dates)

2026-06-22 15:28:27.216 | INFO     | bw_timex.timex_lca:__init__:136 - Initializing TimexLCA object...


2026-06-22 15:28:27.217 | INFO     | bw_timex.timex_lca:__init__:153 - Calculating base LCA...


/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.13/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 1.65e+13)
  warnings.warn(msg, UmfpackWarning)
2026-06-22 15:28:57.297 | INFO     | bw_timex.timex_lca:__init__:170 - Collecting node infos...


Build the timeline **descending into the background**. We use `graph_traversal="bfs"` (recommended
with `traverse_background=True`), a monthly grouping, and start in 2030 so the time-shifts land between
the 2020 and 2030 variants.

> With `traverse_background=True` the traversal descends into the *entire* background supply chain, so
> we cap it with a small `max_calc` to keep this demo fast — the glider chain we care about
> (`P1 → P2 → P3`) is reached in the first few graph-traversal steps.

In [9]:
timeline = tlca.build_timeline(
    starting_datetime="2030-01-01",
    temporal_grouping="month",
    graph_traversal="bfs",
    traverse_background=True,
    max_calc=20,
)
import pandas as pd
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
timeline

2026-06-22 15:29:03.924 | INFO     | bw_timex.timex_lca:build_timeline:349 - Creating activity time mapping...


2026-06-22 15:29:04.294 | INFO     | bw_timex.timeline_builder:__init__:112 - Traversing supply chain graph...


2026-06-22 15:29:04.729 | INFO     | bw_timex.timeline_builder:build_timeline:186 - Building timeline...


,date_producer,producer_name,date_consumer,consumer_name,amount,temporal_market_shares
0,2027-01-01,"market for polyethylene, low density, granulate",2027-01-01,"glider production, passenger car",0.003172,None
1,2027-01-01,market for road vehicle factory,2027-01-01,"glider production, passenger car",0.0,None
2,2027-01-01,"treatment of aluminium scrap, post-consumer, prepared fo...",2027-01-01,"market for aluminium, wrought alloy",0.001931,None
3,2027-01-01,"epoxy resin production, liquid",2027-01-01,"market for epoxy resin, liquid",1.0,None
4,2027-01-01,"market for polyethylene, low density, granulate",2027-01-01,"glider production, passenger car",0.006416,None
...,...,...,...,...,...,...
779,2030-01-01,market for waste glass,2030-01-01,"passenger car production, electric, without battery",-0.036394,None
780,2030-01-01,market for waste glass,2030-01-01,"passenger car production, electric, without battery",-0.0,None
781,2030-01-01,"market for waste rubber, unspecified",2030-01-01,"passenger car production, electric, without battery",-0.018279,None
782,2030-01-01,"market for powertrain, for electric passenger car",2030-01-01,"passenger car production, electric, without battery",0.087365,None


### Double-check: did the background traversal actually reach `P2` and `P3`?

If `traverse_background` works, the timeline must contain rows whose **producer** is the glider market
(`P2`) and the glider production (`P3`) — these live *inside the background* and would never appear with
the default static-background behavior.

The proof is in the **`date_producer`** column: `P2` is pulled back by `TD₁` (−2/−1 years from the 2030
car) and `P3` is pulled back further by `TD₁ ⊗ TD₂` (the −12/−9/−6 month convolution on top). Each
time-shifted cohort is resolved to its temporally-appropriate background variant *directly* — these are
"variant-resolved producers", temporalized in place rather than re-interpolated as temporal markets,
so their `temporal_market_shares` is `None` (that is expected here, not a bug).

In [10]:
producers_traversed = set(timeline["producer_name"])

for name in ["market for glider, passenger car", "glider production, passenger car"]:
    sub = timeline[timeline["producer_name"] == name]
    assert len(sub) > 0, f"MISSING from timeline: {name} (background descent failed!)"
    print(f">>> {name}: {len(sub)} timeline row(s)")
    print(sub[["date_producer", "date_consumer", "consumer_name", "amount"]].to_string(index=False))
    print()

print("OK: background traversal reached both P2 and P3.")

>>> market for glider, passenger car: 4 timeline row(s)
date_producer date_consumer                                       consumer_name    amount
   2028-01-01    2030-01-01 passenger car production, electric, without battery  0.073011
   2028-01-01    2030-01-01 passenger car production, electric, without battery  0.292043
   2029-01-01    2030-01-01 passenger car production, electric, without battery  0.054758
   2029-01-01    2030-01-01 passenger car production, electric, without battery  0.492823

>>> glider production, passenger car: 6 timeline row(s)
date_producer date_consumer                    consumer_name amount
   2027-01-01    2028-01-01 market for glider, passenger car    0.3
   2027-04-01    2028-01-01 market for glider, passenger car    0.4
   2027-07-01    2028-01-01 market for glider, passenger car    0.3
   2028-01-01    2029-01-01 market for glider, passenger car    0.3
   2028-04-01    2029-01-01 market for glider, passenger car    0.4
   2028-07-01    2029-01-01 m

### Step 5 — Compute the inventory and score (time-explicit, background-traversed)

In [11]:
tlca.lci()
tlca.static_lcia()
score_traversed = tlca.static_score
print("traverse_background=True  static score:", score_traversed)

2026-06-22 15:29:05.170 | INFO     | bw_timex.timex_lca:lci:499 - Expanding matrices...


2026-06-22 15:29:06.304 | INFO     | bw_timex.timex_lca:lci:518 - Calculating dynamic inventory...


/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.13/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 1.65e+13)
  warnings.warn(msg, UmfpackWarning)


traverse_background=True  static score: 1.0200287688160934


### Sanity comparison: same system **without** descending into the background

With `traverse_background=False` the glider TDs are ignored and the background is a static snapshot, so
`P2`/`P3` never appear in the timeline. This is a **structural** check (glider chain present vs. absent).

To keep it cheap we **reuse the same `TimexLCA` object** (so we don't pay for a second base LCA) and
only rebuild the *timeline* — no second inventory solve is needed to see that the glider chain is gone.

In [12]:
timeline_static_bg = tlca.build_timeline(
    starting_datetime="2030-01-01",
    temporal_grouping="month",
    graph_traversal="bfs",
    traverse_background=False,
    max_calc=20,
)
producers_static = set(timeline_static_bg["producer_name"])

for name in ["market for glider, passenger car", "glider production, passenger car"]:
    print(f"{name!r}:")
    print(f"    traverse_background=True : {name in producers_traversed}")
    print(f"    traverse_background=False: {name in producers_static}")

2026-06-22 15:29:11.177 | INFO     | bw_timex.timex_lca:build_timeline:328 - No edge filter function provided. Skipping all edges in background databases.


2026-06-22 15:29:16.801 | INFO     | bw_timex.timex_lca:build_timeline:349 - Creating activity time mapping...


2026-06-22 15:29:16.919 | INFO     | bw_timex.timeline_builder:__init__:112 - Traversing supply chain graph...


2026-06-22 15:29:16.930 | INFO     | bw_timex.timeline_builder:build_timeline:186 - Building timeline...


'market for glider, passenger car':
    traverse_background=True : True
    traverse_background=False: False
'glider production, passenger car':
    traverse_background=True : True
    traverse_background=False: False


So the glider chain (`P2`, `P3`) is only present when we descend into the background — confirming the
`traverse_background=True` machinery works end-to-end on real premise data.

> Note we deliberately do **not** compare the two *scores*: with `traverse_background=True` we capped the
> descent at `max_calc=20` for speed, so most of the car's background supply chain is left un-traversed
> and the absolute score is incomplete. For a real analysis you would raise `max_calc` until the score
> stabilizes.

## Cleanup (optional)

Remove the temporal distributions we added to the background exchanges, restoring the premise
databases to their original state.

In [13]:
for e in (e1, e2):
    if "temporal_distribution" in e:
        del e["temporal_distribution"]
        e.save()
print("Removed background TDs.")

Removed background TDs.
